## 1. Auditoria e Tratamento de Dados (ETL)

Durante a inspeção inicial da tabela de `ENTREGAS`, identificou-se um volume significativo de pedidos com chaves duplicadas (`id_pedido`). A simples eliminação destas linhas poderia inflacionar as métricas de sucesso ou ocultar falhas na operação logística. 

Para garantir a integridade dos KPIs, o processo de limpeza foi estruturado através de uma lógica de ordenação categórica, preservando o registo mais crítico de cada pedido através das seguintes regras de negócio:

**Critérios de Priorização (Desempate):**
1. **Rastreabilidade:** Registos com `evento_id` associado têm prioridade absoluta para garantir a correta imputação de custos e dinâmicas.
2. **Visibilidade de Falhas:** O status de `Insucesso` ou `Cancelada` prevalece sobre `Finalizada`.
3. **Padronização de Canal:** O canal `Integrado` tem prioridade sobre a introdução manual na `App`.
4. **Recência:** Como critério final, é mantida a linha com a data e hora (hora_int) mais recentes.

**Resultados da Higienização:**
* **Volume Inicial:** 56.000 linhas
* **Duplicatas Tratadas:** 3.012 conflitos resolvidos
* **Volume Final (Base Tratada):** 52.988 registos únicos

In [9]:
import pandas as pd
from pathlib import Path
import os

In [10]:
raw_data_path = Path("../data/raw/base.xlsx")

try:
    xls = pd.ExcelFile(raw_data_path)
    print("Abas disponíveis:", xls.sheet_names)
except Exception as e:
    print(f"Falha na leitura do arquivo: {e}")

Abas disponíveis: ['ENTREGAS', 'CIDADES', 'EVENTOS', 'MODALIDADES_SLA', 'TABELA_MOTIVOS']


In [11]:
# Carregando as tabelas para DataFrames
print("Carregando bases...")
df_entregas = pd.read_excel(raw_data_path, sheet_name='ENTREGAS')
df_cidades = pd.read_excel(raw_data_path, sheet_name='CIDADES')
df_eventos = pd.read_excel(raw_data_path, sheet_name='EVENTOS')
df_modalidades = pd.read_excel(raw_data_path, sheet_name='MODALIDADES_SLA')
df_motivos = pd.read_excel(raw_data_path, sheet_name='TABELA_MOTIVOS')

# Verificando chaves duplicadas na tabela fato
linhas_iniciais = len(df_entregas)
print(f"Volume inicial (ENTREGAS): {linhas_iniciais} linhas")

duplicados_total = df_entregas.duplicated(subset=['id_pedido'], keep=False).sum()
ids_unicos_duplicados = df_entregas[df_entregas.duplicated(subset=['id_pedido'], keep=False)]['id_pedido'].nunique()

print(f"Total de linhas com conflito: {duplicados_total}")
print(f"Quantidade de id_pedido únicos que estão duplicados: {ids_unicos_duplicados}")

Carregando bases...
Volume inicial (ENTREGAS): 56000 linhas
Total de linhas com conflito: 6024
Quantidade de id_pedido únicos que estão duplicados: 3012


In [12]:
# 1. Garantir que a coluna de data é do tipo datetime para ordenação cronológica correta
df_entregas['data'] = pd.to_datetime(df_entregas['data'])

# 2. Criar uma flag temporária booleana para priorizar quem tem evento_id (True ganha de False)
df_entregas['tem_evento'] = df_entregas['evento_id'].notna()

# 3. Configurar a hierarquia de categorias
status_order = pd.CategoricalDtype(categories=['Cancelada', 'Insucesso', 'Finalizada'], ordered=True)
df_entregas['status'] = df_entregas['status'].astype(status_order)

canal_order = pd.CategoricalDtype(categories=['Integrado', 'App'], ordered=True)
df_entregas['canal'] = df_entregas['canal'].astype(canal_order)

# 4. Ordenar o DataFrame aplicando as regras de negócio em cascata
# id_pedido (agrupa) > tem_evento (desc) > status (asc) > canal (asc) > data (desc) > hora_int (desc)
df_entregas_sorted = df_entregas.sort_values(
    by=['id_pedido', 'tem_evento', 'status', 'canal', 'data', 'hora_int'],
    ascending=[True, False, True, True, False, False] # False = do maior pro menor (True > False, Data mais recente > mais antiga)
)

# 5. Dropar as duplicatas mantendo a primeira ocorrência
df_entregas_clean = df_entregas_sorted.drop_duplicates(subset=['id_pedido'], keep='first').copy()

# 6. Limpar a coluna auxiliar
df_entregas_clean = df_entregas_clean.drop(columns=['tem_evento'])

# Validação Final
linhas_finais = len(df_entregas_clean)
duplicatas_restantes = df_entregas_clean.duplicated(subset=['id_pedido']).sum()

print(f"Volume final após limpeza: {linhas_finais} linhas")
print(f"Total de duplicatas restantes: {duplicatas_restantes}")

Volume final após limpeza: 52988 linhas
Total de duplicatas restantes: 0


C:\Users\Jaime Jr\AppData\Local\Temp\ipykernel_8168\2086506889.py:12: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_entregas['canal'] = df_entregas['canal'].astype(canal_order)


In [13]:
# Exportando a base tratada
processed_path = Path("../data/processed/05_BASE_TRATADA_PYTHON.csv")
df_entregas_clean.to_csv(processed_path, index=False)

print(f"Base tratada exportada para: {processed_path}")

Base tratada exportada para: ..\data\processed\05_BASE_TRATADA_PYTHON.csv
